# Fetch plant natural products from COCONUT

Natural products recorded in COCONUT 2.0 for 12 plants, one CSV per plant in `data/coconut/`.

COCONUT is indexed by organism name, so each plant is searched under its synonyms (rosemary is
listed as *Rosmarinus officinalis*, not *Salvia rosmarinus*) and the results are merged on the
COCONUT identifier. The search returns overlapping pages, so each query is repeated until a
sweep adds no new molecules.

In [ ]:
REFRESH = False

import os
REFRESH = os.environ.get('NL_REFRESH', '1' if REFRESH else '0') == '1'
print('REFRESH =', REFRESH)

In [ ]:
import datetime as dt
import json
import time
from pathlib import Path

import pandas as pd
import requests
from rdkit import Chem, RDLogger

ROOT = Path.cwd().parent if Path.cwd().name == 'src' else Path.cwd()
OUT = ROOT / 'data/coconut'
API = 'https://coconut.naturalproducts.net/api/search'
LIMIT = 100
PAUSE = 0.6
MAX_SWEEPS = 8
STABLE_SWEEPS = 2

RDLogger.DisableLog('rdApp.*')
OUT.mkdir(parents=True, exist_ok=True)
S = requests.Session()
S.headers.update({'User-Agent': 'NeuroLead-EthnoHack2026/1.0', 'Accept': 'application/json'})
print('writing to', OUT)

## The plants

Greek *Sideritis* is collected as a group of four species. The `species` column records which
one each molecule came from.

In [ ]:
PLANTS = {
    'Salvia_rosmarinus':    ('Salvia rosmarinus',    ['Salvia rosmarinus', 'Rosmarinus officinalis']),
    'Crocus_sativus':       ('Crocus sativus',       ['Crocus sativus']),
    'Salvia_fruticosa':     ('Salvia fruticosa',     ['Salvia fruticosa', 'Salvia triloba']),
    'Olea_europaea':        ('Olea europaea',        ['Olea europaea']),
    'Origanum_dictamnus':   ('Origanum dictamnus',   ['Origanum dictamnus', 'Amaracus dictamnus']),
    'Origanum_vulgare':     ('Origanum vulgare',     ['Origanum vulgare', 'Origanum heracleoticum']),
    'Sideritis_greek':      ('Sideritis spp. (Greek)', ['Sideritis scardica', 'Sideritis syriaca',
                                                        'Sideritis euboea', 'Sideritis raeseri']),
    'Hypericum_perforatum': ('Hypericum perforatum', ['Hypericum perforatum']),
    'Thymbra_capitata':      ('Thymbra capitata',      ['Thymbra capitata', 'Thymus capitatus']),
    'Pistacia_lentiscus':    ('Pistacia lentiscus',    ['Pistacia lentiscus']),
    'Melissa_officinalis':   ('Melissa officinalis',   ['Melissa officinalis']),
    'Lavandula_angustifolia':('Lavandula angustifolia', ['Lavandula angustifolia', 'Lavandula officinalis']),
}

FIELDS = ['identifier', 'name', 'canonical_smiles', 'standard_inchi_key', 'iupac_name',
          'murko_framework', 'annotation_level', 'organism_count', 'citation_count',
          'collection_count', 'has_stereo', 'is_parent', 'has_variants', 'parent_id']
print(f'{len(PLANTS)} plants, {sum(len(v[1]) for v in PLANTS.values())} query names')

## Fetching

In [ ]:
def page(query, p):
    body = {'type': 'tags', 'tagType': 'organisms', 'query': query, 'limit': LIMIT, 'page': p}
    for attempt in range(5):
        try:
            r = S.post(API, json=body, timeout=300)
            if r.status_code == 200:
                return r.json().get('data', {})
        except requests.RequestException:
            pass
        time.sleep(2 * (attempt + 1))
    raise RuntimeError(f'COCONUT failed: {query!r} page {p}')


def collect(query):
    seen, reported, quiet, sweeps = {}, None, 0, 0
    while sweeps < MAX_SWEEPS and quiet < STABLE_SWEEPS:
        sweeps += 1
        before, p = len(seen), 1
        while True:
            d = page(query, p)
            reported = d.get('total', reported)
            rows = d.get('data', [])
            for r in rows:
                seen[r['identifier']] = r
            if p >= d.get('last_page', 1) or not rows:
                break
            p += 1
            time.sleep(PAUSE)
        gained = len(seen) - before
        quiet = quiet + 1 if gained == 0 else 0
        print(f'      sweep {sweeps}: +{gained:<4} unique={len(seen):<5} (api total {reported})',
              flush=True)
        time.sleep(PAUSE)
    return list(seen.values()), reported

In [ ]:
retrieved = dt.date.today().isoformat()
manifest = {'source': 'COCONUT 2.0 (https://coconut.naturalproducts.net)',
            'endpoint': 'POST /api/search  type=tags tagType=organisms',
            'retrieved': retrieved,
            'note': ('Paginated search returns overlapping pages, so each name is swept '
                     'repeatedly until two consecutive sweeps add nothing new. The API-reported '
                     'total exceeds the number of distinct molecules returned; both are recorded.'),
            'plants': {}}

for slug, (display, synonyms) in PLANTS.items():
    path = OUT / f'{slug}.csv'
    if path.exists() and not REFRESH:
        d = pd.read_csv(path)
        print(f'{display}: already present, {len(d):,} molecules')
        manifest['plants'][display] = {'file': path.name, 'unique_molecules': int(len(d)),
                                       'skipped': 'file already present'}
        continue

    print(f'=== {display} ===', flush=True)
    frames, per_name = [], {}
    for nm in synonyms:
        print(f'  {nm}', flush=True)
        rows, reported = collect(nm)
        per_name[nm] = {'distinct_returned': len(rows), 'api_reported_total': reported}
        if rows:
            df = pd.DataFrame(rows)
            for c in FIELDS:
                if c not in df.columns:
                    df[c] = None
            df = df[FIELDS].copy()
            df['species'] = nm
            frames.append(df)

    d = pd.concat(frames, ignore_index=True)
    before = len(d)
    d['species'] = d.groupby('identifier')['species'].transform(lambda s: ';'.join(sorted(set(s))))
    d = d.drop_duplicates(subset='identifier').reset_index(drop=True)

    parsed = [Chem.MolFromSmiles(s) if isinstance(s, str) and s else None
              for s in d.canonical_smiles]
    d['smiles_parses'] = [m is not None for m in parsed]
    d['n_fragments'] = [len(Chem.GetMolFrags(m)) if m is not None else 0 for m in parsed]
    d['inchikey_skeleton'] = [k.split('-')[0] if isinstance(k, str) else None
                              for k in d.standard_inchi_key]
    d.insert(0, 'plant', display)
    d.insert(1, 'plant_slug', slug)
    d['source'] = 'COCONUT 2.0'
    d['retrieved_date'] = retrieved
    d = d.sort_values(['is_parent', 'citation_count'],
                      ascending=[False, False]).reset_index(drop=True)
    d.to_csv(path, index=False, encoding='utf-8')

    manifest['plants'][display] = {
        'file': path.name, 'unique_molecules': int(len(d)),
        'distinct_skeletons': int(d.inchikey_skeleton.nunique()),
        'parent_structures': int(d.is_parent.fillna(False).sum()),
        'per_synonym': per_name, 'overlap_between_synonyms': int(before - len(d)),
        'unparseable_smiles': int((~d.smiles_parses).sum()),
        'multi_fragment': int((d.n_fragments > 1).sum())}
    print(f'  -> {path.name}: {len(d):,} molecules, '
          f'{d.inchikey_skeleton.nunique():,} skeletons', flush=True)

(OUT / 'coconut_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('manifest ->', OUT / 'coconut_manifest.json')

## Collected

In [ ]:
rows = []
for f in sorted(OUT.glob('*.csv')):
    d = pd.read_csv(f)
    rows.append({'plant': d.plant.iloc[0], 'molecules': len(d),
                 'skeletons': d.inchikey_skeleton.nunique(),
                 'parents': int(d.is_parent.fillna(False).sum()),
                 'bad_smiles': int((~d.smiles_parses).sum()),
                 'species_in_file': (d.species.str.split(';').explode().nunique()
                                     if 'species' in d.columns else 1)})
summary = pd.DataFrame(rows).sort_values('molecules', ascending=False)
print(summary.to_string(index=False))
print(f'total molecules {summary.molecules.sum():,}   '
      f'distinct skeletons {summary.skeletons.sum():,}')

### Molecules and skeletons

Many entries are stereoisomer or tautomer variants of one structure, so there are more
molecules than distinct skeletons. Notebook 7 collapses them before computing similarity.